# Notebook 10 - Interactive Code_Aster Post-Processor

This notebook loads preserved Code_Aster artifacts and opens the two Tuba review surfaces: inline PyVista mesh/result views and a reviewable web-scene bundle. The default path imports existing solver outputs from `notebooks/code_aster_results/viz_gallery_operating`; set `RUN_CODE_ASTER = True` only after the runtime doctor reports a ready solver.

## 1. Setup

Configure imports, repository paths, and the shared notebook visualization backend.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import HTML, display

from tuba import Model
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime, load_or_run_code_aster_results
from tuba.plotting import plots
from tuba.plotting.notebook import configure_notebook_backend

JUPYTER_BACKEND = configure_notebook_backend()
CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()

## 2. Build The Review Model

The model matches the preserved `viz_gallery_operating` result artifacts.

In [ ]:
model = Model("VizGalleryDemo", standard="ASME_B31.3")

model.add_material(
    "Steel",
    E=2.1e11,
    nu=0.3,
    rho=7850.0,
    alpha=1.2e-5,
    allowable_stress={20.0: 137e6, 150.0: 127e6},
)
model.add_pipe_section("DN100", OD=0.1143, WT=0.00602, corrosion_allowance=0.001)
model.define_load_case(
    "Operating",
    gravity=True,
    pressure=1.5e6,
    temperature=150.0,
    ref_temperature=20.0,
)

with model.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0, 0], support="anchor")
    b.run(3.0)
    b.add_support(type="guide")
    b.bend(radius=0.3, angle=90, plane="XY")
    b.run(2.0)
    b.add_support(type="rest")
    b.bend(radius=0.3, angle=90, plane="XZ")
    b.run(2.0)
    b.end(support="anchor")

model.validate()
print(f"Model: {model.project_name}")
print(f"Nodes: {len(model.nodes)} | Elements: {len(model.elements)} | Supports: {len(model.supports)}")

## 3. Load Code_Aster Results

`RUN_CODE_ASTER = False` imports the preserved result tables. Switching it to `True` executes the solver first, and should only be done after `python -m tuba.solver.code_aster_doctor --check` passes.

In [ ]:
RUN_CODE_ASTER = False
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "viz_gallery_operating"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_RUNTIME.exec_method,
    wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
    docker_image=CODE_ASTER_RUNTIME.docker_image,
)
results = code_aster_run.results
artifact = code_aster_run.artifact

print(f"Result source: {CODE_ASTER_WORK_DIR.resolve()}")
print(f"Solver executed in this run: {code_aster_run.ran_solver}")
print(f"Node results: {len(results.node_results)}")
print(f"Element results: {len(results.element_results)}")
print(f"Analysis mesh nodes: {len(artifact.analysis_mesh.nodes) if artifact.analysis_mesh else 0}")

## 4. Inline Interactive Mesh Views

These cells use the notebook backend selected above. Local notebooks default to embedded HTML; CI can force static output.

In [ ]:
plots.plot_deformed_stress(
    results,
    deform_scale=50.0,
    model=model,
    jupyter_backend=JUPYTER_BACKEND,
)

In [ ]:
plots.plot_displacement_vectors(
    results,
    scale=50.0,
    model=model,
    jupyter_backend=JUPYTER_BACKEND,
)
plots.plot_reactions(
    results,
    scale="auto",
    model=model,
    geometry_opacity=0.25,
    jupyter_backend=JUPYTER_BACKEND,
)

## 5. Export The Reviewable Web Scene

This writes the same result state to the `viewer/` scene-bundle contract.

In [ ]:
from tuba.analysis import create_operating_geometry_state, create_visual_deformed_geometry_state
from tuba.visualization import SceneBuildOptions, build_visualization_scene, write_scene_bundle

operating_state = create_operating_geometry_state(model=model, result_state=artifact.result_state)
visual_state = create_visual_deformed_geometry_state(
    model=model,
    result_state=artifact.result_state,
    visual_scale=50.0,
)
analysis_meshes = [artifact.analysis_mesh] if artifact.analysis_mesh is not None else []
scene = build_visualization_scene(
    model,
    options=SceneBuildOptions(),
    analysis_meshes=analysis_meshes,
    result_states=[artifact.result_state],
    geometry_states=[operating_state, visual_state],
    scene_id="scene:interactive_postprocessor",
)
bundle = write_scene_bundle(scene, REPO_ROOT / "notebooks" / "interactive_postprocessor_bundle")

print(f"Scene bundle: {bundle.root}")
print(f"Scene manifest: {bundle.scene_path}")
print(f"Objects: {len(scene.objects)} | Geometry assets: {len(scene.geometry_assets)} | Overlays: {len(scene.overlays)}")

In [ ]:
display(HTML(f"""
<div style="border:1px solid #c9c3b8; padding:16px; border-radius:8px; background:#fffdf8;">
  <strong>Open the web-scene post-processor locally</strong>
  <pre style="white-space:pre-wrap">cd {REPO_ROOT / 'viewer'}
npm.cmd run dev -- --host 127.0.0.1</pre>
  <p>Then copy <code>{bundle.root}</code> into <code>viewer/public/interactive-postprocessor</code> and open:</p>
  <p><code>http://127.0.0.1:5173/?bundle=interactive-postprocessor</code></p>
</div>
"""))